# Digikala Reviews — Descriptive Analysis

## Objective

This notebook performs descriptive analysis of the Digikala reviews dataset.

The main objectives are:

1. Understand dataset structure and schema.
2. Analyze missing values and data quality.
3. Examine target label distributions.
4. Analyze review text length and repetition.
5. Investigate product-level review distribution.
6. Identify findings that influence downstream modeling decisions.

**Important:** Raw datasets must remain unchanged.
Large CSV files should be processed using Polars LazyFrames.

In [1]:
from pathlib import Path
import polars as pl
import json
import platform
from datetime import datetime

# Resolve project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Dataset paths
COMMENTS_PATH = PROJECT_ROOT / "data" / "raw" / "digikala-comments.csv"
PRODUCTS_PATH = PROJECT_ROOT / "data" / "raw" / "digikala-products.csv"

# Validate paths
assert COMMENTS_PATH.exists(), f"Missing file: {COMMENTS_PATH}"
assert PRODUCTS_PATH.exists(), f"Missing file: {PRODUCTS_PATH}"

print("Project root:", PROJECT_ROOT)
print("Comments size:", round(COMMENTS_PATH.stat().st_size / (1024**3), 3), "GB")
print("Products size:", round(PRODUCTS_PATH.stat().st_size / (1024**3), 3), "GB")
print("Polars version:", pl.__version__)
print("Python version:", platform.python_version())

Project root: /home/jvd/Desktop/persian-nlp-pipeline
Comments size: 1.191 GB
Products size: 0.241 GB
Polars version: 2.1.0
Python version: 3.14.4


In [2]:
comments_lf = pl.scan_csv(
    COMMENTS_PATH,
    infer_schema_length=10000
)

products_lf = pl.scan_csv(
    PRODUCTS_PATH,
    infer_schema_length=10000
)

comments_schema = comments_lf.collect_schema()
products_schema = products_lf.collect_schema()

print("COMMENTS SCHEMA")
print(comments_schema)

print("\nPRODUCTS SCHEMA")
print(products_schema)

COMMENTS SCHEMA
Schema({'id': Int64, 'title': String, 'body': String, 'created_at': String, 'rate': Float64, 'recommendation_status': String, 'is_buyer': Boolean, 'product_id': Int64, 'advantages': String, 'disadvantages': String, 'likes': Int64, 'dislikes': Int64, 'seller_title': String, 'seller_code': String, 'true_to_size_rate': String})

PRODUCTS SCHEMA
Schema({'id': Int64, 'title_fa': String, 'Rate': Int64, 'Rate_cnt': Int64, 'Category1': String, 'Category2': String, 'Brand': String, 'Price': Int64, 'Seller': String, 'Is_Fake': Boolean, 'min_price_last_month': Int64, 'sub_category': String})


In [3]:
comments_sample = comments_lf.head(10).collect()
products_sample = products_lf.head(10).collect()

display(comments_sample)
display(products_sample)

id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
i64,str,str,str,f64,str,bool,i64,str,str,i64,i64,str,str,str
53672599,"""پیشنهاد نمیشود""","""به درد نمیخوره ""","""23 شهریور 1402""",1.0,"""not_recommended""",true,252058,"""nan""","""nan""",0,0,"""دیجی‌کالا""","""5A52N""","""nan"""
9897229,"""بسته بندی بد""","""می‌تونست به عنوان یه کالای فره…","""16 تیر 1399""",0.0,"""recommended""",true,252058,"""['تجربه جالبی بود برام ']""","""['بسته بندی جالبی نداشت']""",1,0,"""دیجی‌کالا""","""5A52N""","""nan"""
38074516,"""برس ریمل""","""بسته بندیش خوب بود کاربرد و …","""26 مرداد 1401""",0.0,"""recommended""",true,3331597,"""nan""","""nan""",0,0,"""آرالیا بیوتی""","""ADM47""","""nan"""
18628562,"""خوبه و خوشرنگ""","""به نظرم خوبه فقط یکم ظریفه. از…","""28 اسفند 1399""",0.0,"""recommended""",true,3331329,"""nan""","""nan""",0,0,"""اینجاست آ ""","""9ZMCZ""","""nan"""
53301258,"""برس رنگ مو""","""معمولیه اگه واسه خونه رنگ کردن…","""12 شهریور 1402""",3.0,"""recommended""",true,3255700,"""nan""","""nan""",0,0,"""گالری آرایشی به سیما""","""CDWHA""","""nan"""
53266157,"""خوبه""","""قبلا هم استفاده کردم اگه بلد ب…","""11 شهریور 1402""",5.0,"""recommended""",true,3305270,"""nan""","""nan""",0,0,"""یانگوم""","""ERKSP""","""nan"""
46206235,"""بدک نیست ""","""خوبه""","""8 اسفند 1401""",2.0,"""nan""",true,3480048,"""nan""","""nan""",0,0,"""بازرگانی سوشیان""","""CKWY4""","""nan"""
40597171,"""پد شستشوی براش""","""خیلی به کارتون میاد قیمتشم منا…","""24 مهر 1401""",5.0,"""recommended""",true,3480048,"""['جنسش خوبه\r', 'خوش رنگه\r', …","""['نداره ']""",1,0,"""بازرگانی سوشیان""","""CKWY4""","""nan"""
46198656,"""لنز ناخون""","""دقیقا مطابق عکس بود""","""7 اسفند 1401""",5.0,"""recommended""",true,7626372,"""nan""","""nan""",0,0,"""هستی نیل""","""E2C2D""","""nan"""


id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
i64,str,i64,i64,str,str,str,i64,str,bool,i64,str
7096438,"""آبسلانگ مدل s5 بسته 250 عددی""",90,4,"""آبسلانگ""",null,"""متفرقه""",634800,"""سلامت ساز راد""",false,0,"""beauty"""
2845119,""" آبسلانگ مدل M-1 بسته 400 عددی""",84,217,"""آبسلانگ""",null,"""متفرقه""",818800,"""مهر افزون""",false,0,"""beauty"""
6117745,"""آبسلانگ مدل m50 مجموعه 500 عدد…",88,14,"""آبسلانگ""",null,"""متفرقه""",920000,"""یانگوم""",false,0,"""beauty"""
1912926,"""استند ابسلانگ مدل S01""",78,25,"""آبسلانگ""",null,"""متفرقه""",1100000,"""بهراد طب بیدار""",false,0,"""beauty"""
6335462,"""آبسلانگ نوری تسلامد مدل All-in…",84,25,"""آبسلانگ""",null,"""تسلا مد""",1530000,"""دیجی‌کالا""",false,0,"""beauty"""
6335562,"""سری آبسلانگ تسلامد مدل All-In-…",80,6,"""آبسلانگ""",null,"""تسلا مد""",1521000,"""دیجی‌کالا""",false,0,"""beauty"""
7096725,"""آبسلانگ مدل s20 بسته 1000 عددی""",84,6,"""آبسلانگ""",null,"""متفرقه""",1889500,"""سلامت ساز راد""",false,0,"""beauty"""
7096768,"""آبسلانگ مدل s10 بسته 500 عددی""",100,2,"""آبسلانگ""",null,"""متفرقه""",1872000,"""سلامت ساز راد""",false,0,"""beauty"""
6005981,"""استند آبسلانگ گلدن بیوتی مدل 0…",84,5,"""آبسلانگ""",null,"""متفرقه""",1940000,"""گلد بیوتی سنتر""",false,0,"""beauty"""


## 1. Dataset Overview

This section examines the overall size and structure of both datasets.

All row counts and schema information are computed from the full datasets.

In [4]:
comments_count = comments_lf.select(
    pl.len().alias("total_rows")
).collect()

products_count = products_lf.select(
    pl.len().alias("total_rows")
).collect()

print("Total comments:", comments_count.item())
print("Total products:", products_count.item())

Total comments: 6156289
Total products: 1283496


In [5]:
def missing_values_report(lf: pl.LazyFrame) -> pl.DataFrame:
    total_rows = lf.select(pl.len()).collect().item()

    report = (
        lf.select([
            pl.col(column).null_count().alias(column)
            for column in lf.collect_schema().names()
        ])
        .collect()
        .unpivot(
            variable_name="column",
            value_name="missing_count"
        )
        .with_columns(
            (pl.col("missing_count") / total_rows * 100)
            .round(2)
            .alias("missing_percentage")
        )
        .sort("missing_percentage", descending=True)
    )

    return report


comments_missing = missing_values_report(comments_lf)
products_missing = missing_values_report(products_lf)

print("Comments — Missing Values")
display(comments_missing)

print("Products — Missing Values")
display(products_missing)

Comments — Missing Values


column,missing_count,missing_percentage
str,u32,f64
"""id""",0,0.0
"""title""",0,0.0
"""body""",0,0.0
"""created_at""",0,0.0
"""rate""",0,0.0
…,…,…
"""likes""",0,0.0
"""dislikes""",0,0.0
"""seller_title""",0,0.0


Products — Missing Values


column,missing_count,missing_percentage
str,u32,f64
"""Category2""",210108,16.37
"""id""",0,0.0
"""title_fa""",0,0.0
"""Rate""",0,0.0
"""Rate_cnt""",0,0.0
…,…,…
"""Price""",0,0.0
"""Seller""",0,0.0
"""Is_Fake""",0,0.0


In [6]:
recommendation_distribution = (
    comments_lf
    .group_by("recommendation_status")
    .agg(
        pl.len().alias("count")
    )
    .with_columns(
        (
            pl.col("count") /
            pl.col("count").sum() * 100
        ).round(2).alias("percentage")
    )
    .sort("count", descending=True)
    .collect()
)

display(recommendation_distribution)

recommendation_status,count,percentage
str,u32,f64
"""recommended""",4162839,67.62
"""nan""",894426,14.53
"""no_idea""",594121,9.65
"""not_recommended""",504903,8.2


In [7]:
rating_distribution = (
    comments_lf
    .with_columns(
        pl.when(pl.col("rate").is_null())
        .then(pl.lit(None, dtype=pl.String))
        .when(pl.col("rate") < 2.5)
        .then(pl.lit("low"))
        .when(pl.col("rate") < 4.0)
        .then(pl.lit("medium"))
        .when(pl.col("rate") <= 5.0)
        .then(pl.lit("high"))
        .otherwise(pl.lit("invalid"))
        .alias("rating_band")
    )
    .group_by("rating_band")
    .agg(pl.len().alias("count"))
    .with_columns(
        (
            pl.col("count") /
            pl.col("count").sum() * 100
        ).round(2).alias("percentage")
    )
    .sort("count", descending=True)
    .collect()
)

display(rating_distribution)

rating_band,count,percentage
str,u32,f64
"""high""",4004905,65.05
"""low""",1092329,17.74
"""medium""",1059054,17.2
"""invalid""",1,0.0


In [8]:
text_samples = (
    comments_lf
    .select(["id", "title", "body", "rate", "recommendation_status"])
    .head(20)
    .collect()
)

display(text_samples)

id,title,body,rate,recommendation_status
i64,str,str,f64,str
53672599,"""پیشنهاد نمیشود""","""به درد نمیخوره """,1.0,"""not_recommended"""
9897229,"""بسته بندی بد""","""می‌تونست به عنوان یه کالای فره…",0.0,"""recommended"""
38074516,"""برس ریمل""","""بسته بندیش خوب بود کاربرد و …",0.0,"""recommended"""
18628562,"""خوبه و خوشرنگ""","""به نظرم خوبه فقط یکم ظریفه. از…",0.0,"""recommended"""
53301258,"""برس رنگ مو""","""معمولیه اگه واسه خونه رنگ کردن…",3.0,"""recommended"""
…,…,…,…,…
51783725,"""خیلی خوب بود""","""خیلی عالی بود ، کاربردیه ، دیگ…",4.0,"""recommended"""
52776060,"""اصلا بدرد نمیخوره نخرید""","""بدرد نخور واقعی وقتی بدستم رسی…",1.0,"""not_recommended"""
52767482,"""برس ابرو""","""کمی گران اما با کیفیت""",5.0,"""recommended"""


In [9]:
text_samples = (
    comments_lf
    .select(["id", "title", "body", "rate", "recommendation_status"])
    .head(20)
    .collect()
)

display(text_samples)

id,title,body,rate,recommendation_status
i64,str,str,f64,str
53672599,"""پیشنهاد نمیشود""","""به درد نمیخوره """,1.0,"""not_recommended"""
9897229,"""بسته بندی بد""","""می‌تونست به عنوان یه کالای فره…",0.0,"""recommended"""
38074516,"""برس ریمل""","""بسته بندیش خوب بود کاربرد و …",0.0,"""recommended"""
18628562,"""خوبه و خوشرنگ""","""به نظرم خوبه فقط یکم ظریفه. از…",0.0,"""recommended"""
53301258,"""برس رنگ مو""","""معمولیه اگه واسه خونه رنگ کردن…",3.0,"""recommended"""
…,…,…,…,…
51783725,"""خیلی خوب بود""","""خیلی عالی بود ، کاربردیه ، دیگ…",4.0,"""recommended"""
52776060,"""اصلا بدرد نمیخوره نخرید""","""بدرد نخور واقعی وقتی بدستم رسی…",1.0,"""not_recommended"""
52767482,"""برس ابرو""","""کمی گران اما با کیفیت""",5.0,"""recommended"""


## 2. Missing Values and Empty Data

This section distinguishes between:
- Actual null values
- Empty strings
- Whitespace-only strings
- String placeholders such as "nan", "null", and "none"

The purpose is to identify missing values that may not be recognized as null during CSV parsing.

In [10]:
def missing_like_report(lf: pl.LazyFrame) -> pl.DataFrame:
    schema = lf.collect_schema()
    string_columns = [
        name for name, dtype in schema.items()
        if dtype == pl.String
    ]

    expressions = []

    for column in string_columns:
        value = pl.col(column).str.strip_chars().str.to_lowercase()

        expressions.append(
            pl.col(column)
            .is_null()
            .sum()
            .alias(f"{column}__null")
        )

        expressions.append(
            (value == "")
            .fill_null(False)
            .sum()
            .alias(f"{column}__empty")
        )

        expressions.append(
            value.is_in(["nan", "null", "none", "n/a"])
            .fill_null(False)
            .sum()
            .alias(f"{column}__placeholder")
        )

    counts = lf.select(expressions).collect()

    records = []

    for column in string_columns:
        records.append({
            "column": column,
            "null_count": counts[f"{column}__null"].item(),
            "empty_count": counts[f"{column}__empty"].item(),
            "placeholder_count": counts[f"{column}__placeholder"].item(),
        })

    return pl.DataFrame(records)


comments_missing_like = missing_like_report(comments_lf)

display(
    comments_missing_like.sort(
        "placeholder_count",
        descending=True
    )
)

column,null_count,empty_count,placeholder_count
str,i64,i64,i64
"""true_to_size_rate""",0,0,6065516
"""disadvantages""",0,0,5744017
"""advantages""",0,0,5448084
"""title""",0,1353,2865062
"""recommendation_status""",0,0,894426
"""seller_title""",0,0,302181
"""seller_code""",0,0,302181
"""body""",0,0,637
"""created_at""",0,0,0


In [11]:
invalid_rates = (
    comments_lf
    .filter(
        (pl.col("rate") < 0) |
        (pl.col("rate") > 5)
    )
    .group_by("rate")
    .agg(pl.len().alias("count"))
    .collect()
)

display(invalid_rates)

rate,count
f64,u32
2500.0,1


In [12]:
label_values = (
    comments_lf
    .group_by("recommendation_status")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

display(label_values)

recommendation_status,count
str,u32
"""recommended""",4162839
"""nan""",894426
"""no_idea""",594121
"""not_recommended""",504903


In [13]:
invalid_rate_record = (
    comments_lf
    .filter(pl.col("rate") > 5)
    .select([
        "id",
        "title",
        "body",
        "rate",
        "recommendation_status",
        "product_id"
    ])
    .collect()
)

display(invalid_rate_record)

id,title,body,rate,recommendation_status,product_id
i64,str,str,f64,str,i64
38631814,"""زیبا""","""کوچک و با کیفیت""",2500.0,"""recommended""",5095301


## 3. Review Text Length Analysis

This section analyzes the character and word length of review bodies.

Statistics are computed over the full dataset, excluding placeholder values
such as "nan" from meaningful text-length calculations.

The analysis includes:
- Character length distribution
- Word count distribution
- Mean, median, standard deviation
- 75th, 90th, 95th, 99th and 99.9th percentiles

In [14]:
body_clean = pl.col("body").str.strip_chars()

text_lf = (
    comments_lf
    .filter(
        body_clean.is_not_null()
        & (body_clean != "")
        & (
            body_clean
            .str.to_lowercase()
            .is_in(["nan", "null", "none", "n/a"])
            .not_()
        )
    )
    .with_columns([
        body_clean.str.len_chars().alias("char_length"),
        body_clean.str.count_matches(r"\S+").alias("word_count")
    ])
)

In [15]:
text_quality_check = text_lf.select([
    pl.len().alias("valid_text_count"),
    (pl.col("char_length") == 4000)
        .sum()
        .alias("texts_exactly_4000_chars"),
    (pl.col("char_length") >= 3500)
        .sum()
        .alias("texts_at_least_3500_chars"),
    (pl.col("char_length") <= 10)
        .sum()
        .alias("texts_at_most_10_chars"),
]).collect()

display(text_quality_check)

valid_text_count,texts_exactly_4000_chars,texts_at_least_3500_chars,texts_at_most_10_chars
u32,u32,u32,u32
6155652,6,18,825364


In [16]:
text_by_class = (
    text_lf
    .join(
        comments_lf.select([
            "id",
            "recommendation_status"
        ]),
        on="id",
        how="left"
    )
    .with_columns(
        pl.when(
            pl.col("recommendation_status")
            .is_in(["nan", "null", "none", "n/a"])
        )
        .then(None)
        .otherwise(pl.col("recommendation_status"))
        .alias("recommendation_status")
    )
    .group_by("recommendation_status")
    .agg([
        pl.len().alias("review_count"),
        pl.col("char_length").mean().round(2).alias("char_mean"),
        pl.col("char_length").median().alias("char_median"),
        pl.col("char_length").quantile(0.90).alias("char_p90"),
        pl.col("word_count").mean().round(2).alias("word_mean"),
        (pl.col("char_length") <= 10)
            .sum()
            .alias("short_reviews_count")
    ])
    .sort("review_count", descending=True)
    .collect()
)

display(text_by_class)

recommendation_status,review_count,char_mean,char_median,char_p90,word_mean,short_reviews_count
str,u32,f64,f64,f64,f64,u32
"""recommended""",4166573,44.07,27.0,95.0,9.24,547682
null,894996,46.35,25.0,105.0,9.7,205953
"""no_idea""",594886,59.18,37.0,129.0,12.37,51479
"""not_recommended""",505661,73.21,48.0,160.0,15.18,21011


In [17]:
body_clean = pl.col("body").str.strip_chars()
status_clean = pl.col("recommendation_status").str.strip_chars()

text_by_class = (
    comments_lf
    .filter(
        body_clean.is_not_null()
        & (body_clean != "")
        & (
            body_clean.str.to_lowercase()
            .is_in(["nan", "null", "none", "n/a"])
            .not_()
        )
    )
    .with_columns([
        body_clean.str.len_chars().alias("char_length"),
        body_clean.str.count_matches(r"\S+").alias("word_count"),
        pl.when(
            status_clean.is_null()
            | (status_clean == "")
            | status_clean.str.to_lowercase().is_in(
                ["nan", "null", "none", "n/a"]
            )
        )
        .then(None)
        .otherwise(status_clean)
        .alias("clean_status")
    ])
    .group_by("clean_status")
    .agg([
        pl.len().alias("review_count"),
        pl.col("char_length").mean().round(2).alias("char_mean"),
        pl.col("char_length").median().alias("char_median"),
        pl.col("char_length").quantile(0.90).alias("char_p90"),
        pl.col("word_count").mean().round(2).alias("word_mean"),
        (pl.col("char_length") <= 10)
            .sum()
            .alias("short_reviews_count")
    ])
    .sort("review_count", descending=True)
    .collect()
)

display(text_by_class)

clean_status,review_count,char_mean,char_median,char_p90,word_mean,short_reviews_count
str,u32,f64,f64,f64,f64,u32
"""recommended""",4162417,44.04,27.0,95.0,9.23,547122
null,894290,46.3,25.0,104.0,9.69,205833
"""no_idea""",594078,59.13,37.0,129.0,12.36,51419
"""not_recommended""",504867,73.14,48.0,159.0,15.17,20990


In [18]:
display(
    text_by_class.with_columns(
        (
            pl.col("short_reviews_count")
            / pl.col("review_count")
            * 100
        )
        .round(2)
        .alias("short_reviews_percentage")
    )
)

clean_status,review_count,char_mean,char_median,char_p90,word_mean,short_reviews_count,short_reviews_percentage
str,u32,f64,f64,f64,f64,u32,f64
"""recommended""",4162417,44.04,27.0,95.0,9.23,547122,13.14
null,894290,46.3,25.0,104.0,9.69,205833,23.02
"""no_idea""",594078,59.13,37.0,129.0,12.36,51419,8.66
"""not_recommended""",504867,73.14,48.0,159.0,15.17,20990,4.16


In [19]:
product_review_stats = (
    comments_lf
    .group_by("product_id")
    .agg(
        pl.len().alias("review_count")
    )
    .select([
        pl.len().alias("unique_products"),
        pl.col("review_count").min().alias("min_reviews"),
        pl.col("review_count").median().alias("median_reviews"),
        pl.col("review_count").mean().round(2).alias("mean_reviews"),
        pl.col("review_count").quantile(0.90).alias("p90_reviews"),
        pl.col("review_count").quantile(0.95).alias("p95_reviews"),
        pl.col("review_count").quantile(0.99).alias("p99_reviews"),
        pl.col("review_count").max().alias("max_reviews"),
        (pl.col("review_count") == 1)
            .sum()
            .alias("single_review_products")
    ])
    .collect()
)

display(product_review_stats)

unique_products,min_reviews,median_reviews,mean_reviews,p90_reviews,p95_reviews,p99_reviews,max_reviews,single_review_products
u32,u32,f64,f64,f64,f64,f64,u32,u32
331599,1,4.0,18.57,46.0,110.0,200.0,400,91944


In [20]:
display(
    text_by_class.with_columns(
        (
            pl.col("short_reviews_count")
            / pl.col("review_count")
            * 100
        )
        .round(2)
        .alias("short_reviews_percentage")
    )
)

clean_status,review_count,char_mean,char_median,char_p90,word_mean,short_reviews_count,short_reviews_percentage
str,u32,f64,f64,f64,f64,u32,f64
"""recommended""",4162417,44.04,27.0,95.0,9.23,547122,13.14
null,894290,46.3,25.0,104.0,9.69,205833,23.02
"""no_idea""",594078,59.13,37.0,129.0,12.36,51419,8.66
"""not_recommended""",504867,73.14,48.0,159.0,15.17,20990,4.16


In [21]:
short_reviews = (
    text_lf
    .filter(pl.col("char_length") <= 10)
    .select(["id", "body", "char_length", "word_count"])
    .head(20)
    .collect()
)

long_reviews = (
    text_lf
    .sort("char_length", descending=True)
    .select(["id", "body", "char_length", "word_count"])
    .head(20)
    .collect()
)

print("Short reviews — first 20")
display(short_reviews)

print("Longest reviews — top 20")
display(long_reviews)

Short reviews — first 20


id,body,char_length,word_count
i64,str,u32,u32
46206235,"""خوبه""",4,1
48786928,"""خوب""",3,1
7993881,"""خوب بود""",7,2
32054727,"""خوب بود""",7,2
20624535,"""خوبه""",4,1
…,…,…,…
35067728,"""معمولي بود""",10,2
52940103,"""خوب بود""",7,2
46736690,"""خوبع""",4,1


Longest reviews — top 20


id,body,char_length,word_count
i64,str,u32,u32
22914796,"""سال ۲۰۰۷ بود که Activision و I…",4000,795
37876528,"""من یه پسر ۱۵ ساله خیلی خوشگل ن…",4000,908
40951601,"""برای خرید یه ماشین اصلاح خوب و…",4000,862
43932524,"""تصمیم خرید کنسول برای منِ 32 س…",4000,854
38287196,"""هر چیزی که در پایین گفتم واقعن…",4000,825
…,…,…,…
12437956,"""از نظر رنگ و شکل مثل عکسشه و ت…",3639,783
35193095,"""هنوز استفاده نکردم ولی ترجمه س…",3533,740
4156731,"""نجات جهان کپی ایرانی بازی پندم…",3521,673


In [22]:
# تعداد نظرات هر محصول؛ فقط product_id معتبر
product_counts = (
    comments_lf
    .filter(pl.col("product_id").is_not_null())
    .group_by("product_id")
    .agg(pl.len().alias("review_count"))
    .sort("review_count", descending=True)
    .collect()
)

total_reviews = product_counts["review_count"].sum()

concentration_report = pl.DataFrame({
    "group": ["Top 10", "Top 100", "Top 1000"],
    "review_count": [
        product_counts.head(10)["review_count"].sum(),
        product_counts.head(100)["review_count"].sum(),
        product_counts.head(1000)["review_count"].sum()
    ]
}).with_columns(
    (
        pl.col("review_count") / total_reviews * 100
    ).round(2).alias("percentage_of_reviews")
)

product_id_quality = comments_lf.select([
    pl.len().alias("total_comment_rows"),
    pl.col("product_id").null_count().alias("null_product_ids"),
    pl.col("product_id").n_unique().alias("unique_product_ids_including_null")
]).collect()

display(concentration_report)
display(product_id_quality)

group,review_count,percentage_of_reviews
str,i64,f64
"""Top 10""",2886,0.05
"""Top 100""",21912,0.36
"""Top 1000""",204391,3.32


total_comment_rows,null_product_ids,unique_product_ids_including_null
u32,u32,u32
6156289,0,331599


In [23]:
product_id_check = (
    comments_lf
    .select("product_id")
    .filter(pl.col("product_id").is_not_null())
    .unique()
    .join(
        products_lf.select(pl.col("id").alias("product_id")).unique(),
        on="product_id",
        how="anti"
    )
    .select(pl.len().alias("unmatched_product_ids"))
    .collect()
)

display(product_id_check)

unmatched_product_ids
u32
0


In [24]:
seller_stats = (
    comments_lf
    .with_columns(
        pl.when(
            pl.col("seller_code").is_null()
            | pl.col("seller_code").str.strip_chars().is_in(
                ["", "nan", "null", "none", "n/a"]
            )
        )
        .then(None)
        .otherwise(pl.col("seller_code").str.strip_chars())
        .alias("clean_seller_code")
    )
    .group_by("clean_seller_code")
    .agg(
        pl.len().alias("review_count")
    )
    .filter(pl.col("clean_seller_code").is_not_null())
    .sort("review_count", descending=True)
    .collect()
)

total_seller_reviews = seller_stats["review_count"].sum()

seller_report = pl.DataFrame({
    "group": ["Top 10", "Top 100", "Top 1000"],
    "review_count": [
        seller_stats.head(10)["review_count"].sum(),
        seller_stats.head(100)["review_count"].sum(),
        seller_stats.head(1000)["review_count"].sum()
    ]
}).with_columns(
    (
        pl.col("review_count") / total_seller_reviews * 100
    ).round(2).alias("percentage_of_reviews")
)

seller_quality = pl.DataFrame({
    "unique_sellers": [seller_stats.height],
    "total_reviews_with_valid_seller": [total_seller_reviews],
    "reviews_without_valid_seller": [
        6156289 - total_seller_reviews
    ]
})

display(seller_report)
display(seller_quality)

group,review_count,percentage_of_reviews
str,i64,f64
"""Top 10""",617089,10.54
"""Top 100""",1538055,26.27
"""Top 1000""",3773227,64.45


unique_sellers,total_reviews_with_valid_seller,reviews_without_valid_seller
i64,i64,i64
29213,5854108,302181


In [25]:
duplicate_ids = (
    comments_lf
    .group_by("id")
    .agg(
        pl.len().alias("occurrences")
    )
    .filter(pl.col("occurrences") > 1)
    .select([
        pl.len().alias("duplicated_id_count"),
        (pl.col("occurrences") - 1)
            .sum()
            .alias("extra_duplicate_rows"),
        pl.col("occurrences").max()
            .alias("max_occurrences_per_id")
    ])
    .collect()
)

display(duplicate_ids)

duplicated_id_count,extra_duplicate_rows,max_occurrences_per_id
u32,u32,u32
3226,3229,3


In [26]:
duplicate_id_details = (
    comments_lf
    .join(
        comments_lf
        .group_by("id")
        .agg(pl.len().alias("occurrences"))
        .filter(pl.col("occurrences") > 1)
        .select("id", "occurrences"),
        on="id",
        how="inner"
    )
    .select([
        "id",
        "occurrences",
        "title",
        "body",
        "rate",
        "recommendation_status",
        "product_id",
        "seller_code"
    ])
    .sort("id")
    .collect()
)

display(duplicate_id_details.head(30))

id,occurrences,title,body,rate,recommendation_status,product_id,seller_code
i64,u32,str,str,f64,str,i64,str
1002211,2,"""پیشنهاد""","""یکی از قشنگترین داستانهای کودک…",3.0,"""nan""",65742,"""nan"""
1002211,2,"""پیشنهاد""","""یکی از قشنگترین داستانهای کودک…",3.0,"""nan""",65742,"""nan"""
1123478,2,"""کتاب صوتی شازده کوچولو""","""یکی از مهمترین مزایای کتاب صوت…",0.0,"""nan""",156134,"""nan"""
1123478,2,"""کتاب صوتی شازده کوچولو""","""یکی از مهمترین مزایای کتاب صوت…",0.0,"""nan""",156134,"""nan"""
1153354,2,"""کیفیت عالی اثر""","""تنها میتونم بگم این کتاب فوق ا…",5.0,"""nan""",68722,"""nan"""
…,…,…,…,…,…,…,…
1237215,2,"""موثر و مفید""","""خیلی خوب دوبله شده گوینده بسی…",3.0,"""nan""",170832,"""nan"""
1271444,2,"""خوب نیست""","""من 1 بار گرفتم خیلی پوستو سفید…",3.0,"""nan""",82046,"""nan"""
1271444,2,"""خوب نیست""","""من 1 بار گرفتم خیلی پوستو سفید…",3.0,"""nan""",82046,"""nan"""


In [27]:
all_columns = comments_lf.collect_schema().names()

exact_duplicate_groups = (
    comments_lf
    .group_by(all_columns)
    .agg(
        pl.len().alias("occurrences")
    )
    .filter(pl.col("occurrences") > 1)
    .select([
        pl.len().alias("exact_duplicate_groups"),
        (pl.col("occurrences") - 1)
            .sum()
            .alias("extra_exact_duplicate_rows"),
        pl.col("occurrences").max()
            .alias("max_exact_occurrences")
    ])
    .collect()
)

display(exact_duplicate_groups)

exact_duplicate_groups,extra_exact_duplicate_rows,max_exact_occurrences
u32,u32,u32
1318,1318,2


In [28]:
duplicate_columns = [
    col for col in all_columns if col != "id"
]

duplicate_id_profiles = (
    comments_lf
    .group_by("id")
    .agg([
        pl.col(col).n_unique().alias(f"{col}_nunique")
        for col in duplicate_columns
    ])
    .filter(
        pl.any_horizontal([
            pl.col(f"{col}_nunique") > 1
            for col in duplicate_columns
        ])
    )
)

# تعداد شناسه‌های تکراری که در هر ستون تفاوت دارند
different_columns_report = duplicate_id_profiles.select([
    (pl.col(f"{col}_nunique") > 1)
    .sum()
    .alias(col)
    for col in duplicate_columns
]).collect()

display(different_columns_report)

title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
85,241,0,79,101,22,0,41,6,1217,294,154,147,4


In [29]:
different_id_samples = (
    comments_lf
    .join(
        duplicate_id_profiles.select("id"),
        on="id",
        how="semi"
    )
    .sort("id")
    .select(all_columns)
    .collect()
)

display(different_id_samples.head(20))

id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
i64,str,str,str,f64,str,bool,i64,str,str,i64,i64,str,str,str
1271444,"""خوب نیست""","""من 1 بار گرفتم خیلی پوستو سفید…","""25 اسفند 1395""",3.0,"""nan""",false,82046,"""nan""","""nan""",9,3,"""nan""","""nan""","""nan"""
1271444,"""خوب نیست""","""من 1 بار گرفتم خیلی پوستو سفید…","""25 اسفند 1395""",3.0,"""nan""",false,82046,"""nan""","""nan""",8,3,"""nan""","""nan""","""nan"""
1542527,"""دوست پوست های مستعد آکنه""","""مشکل پوست های مستعد آکنه اینه …","""9 آبان 1396""",2.5,"""nan""",false,82046,"""['سبک', 'حجم مناسب ', 'پوست رو…","""['فاکتور حفاظتی کم SPf 30']""",8,1,"""nan""","""nan""","""nan"""
1542527,"""دوست پوست های مستعد آکنه""","""مشکل پوست های مستعد آکنه اینه …","""9 آبان 1396""",2.5,"""nan""",false,82046,"""['سبک', 'حجم مناسب ', 'پوست رو…","""['فاکتور حفاظتی کم SPf 30']""",8,0,"""nan""","""nan""","""nan"""
1766600,"""ژل حجم دهنده""","""حتماً پیشنهاد میکنم عالیه فقط …","""5 فروردین 1397""",5.0,"""nan""",false,524441,"""['تاثیر خوبی داره ', 'طول و قط…","""['ندارد ']""",50,20,"""nan""","""nan""","""nan"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3255644,"""بسیاربد وبی اثر""","""کلا هیچ تأثیری ندارن..من موهام…","""1 مرداد 1398""",0.25,"""nan""",false,82080,"""['ندارد']""","""['زیاد']""",8,1,"""nan""","""nan""","""nan"""
4083940,"""باعث جوش میشه""","""من از این لوسیون استفاده کردم …","""13 مهر 1398""",1.5,"""nan""",false,82046,"""nan""","""nan""",3,0,"""nan""","""nan""","""nan"""
4083940,"""باعث جوش میشه""","""من از این لوسیون استفاده کردم …","""13 مهر 1398""",1.5,"""nan""",false,82046,"""nan""","""nan""",2,0,"""nan""","""nan""","""nan"""


In [30]:
conflict_summary = (
    duplicate_id_profiles
    .select([
        (pl.col("body_nunique") > 1)
            .sum().alias("different_body_ids"),

        (pl.col("recommendation_status_nunique") > 1)
            .sum().alias("different_recommendation_ids"),

        (pl.col("rate_nunique") > 1)
            .sum().alias("different_rate_ids"),

        (
            (pl.col("body_nunique") > 1)
            & (pl.col("recommendation_status_nunique") > 1)
        )
        .sum().alias("body_and_label_conflicts")
    ])
    .collect()
)

display(conflict_summary)

different_body_ids,different_recommendation_ids,different_rate_ids,body_and_label_conflicts
u32,u32,u32,u32
241,101,79,50


In [31]:
label_conflict_samples = (
    duplicate_id_profiles
    .filter(pl.col("recommendation_status_nunique") > 1)
    .select("id")
    .join(
        comments_lf,
        on="id",
        how="inner"
    )
    .select([
        "id",
        "body",
        "rate",
        "recommendation_status",
        "created_at",
        "product_id"
    ])
    .sort("id")
    .collect()
)

display(label_conflict_samples.head(20))

id,body,rate,recommendation_status,created_at,product_id
i64,str,f64,str,str,i64
24565793,"""قدرت چسبندگی خوبی داره، بد نیس…",3.0,"""no_idea""","""6 شهریور 1400""",2233238
24565793,"""قدرت چسبندگی خوبی داره، عملکرد…",5.0,"""recommended""","""6 شهریور 1400""",2233238
32424494,"""راضی هستیم""",4.0,"""recommended""","""15 اسفند 1400""",3262588
32424494,"""راضی هستیم""",4.0,"""nan""","""15 اسفند 1400""",3262588
33691371,"""کرم خوبیه اما تاریخ مصرفش گذشت…",4.0,"""not_recommended""","""21 فروردین 1401""",3548526
…,…,…,…,…,…
37394585,"""با این که ایرانی هست اما عالیه""",5.0,"""recommended""","""7 مرداد 1401""",2233954
38142883,"""غیر اصل ...رقیق و تیره و کاورد…",1.0,"""not_recommended""","""27 مرداد 1401""",6028466
38142883,"""عالی""",5.0,"""nan""","""27 مرداد 1401""",6028466


In [32]:
label_conflict_audit = (
    comments_lf
    .with_columns(
        pl.when(
            pl.col("recommendation_status")
            .str.strip_chars()
            .str.to_lowercase()
            .is_in(["nan", "null", "none", "n/a", ""])
        )
        .then(None)
        .otherwise(
            pl.col("recommendation_status").str.strip_chars()
        )
        .alias("clean_status")
    )
    .group_by("id")
    .agg([
        pl.len().alias("row_count"),
        pl.col("clean_status").drop_nulls()
            .n_unique().alias("valid_label_count"),
        pl.col("clean_status").is_null()
            .any().alias("has_missing_label"),
        pl.col("clean_status").drop_nulls()
            .unique().alias("valid_labels")
    ])
    .filter(pl.col("row_count") > 1)
)

label_conflict_summary = label_conflict_audit.select([
    (pl.col("valid_label_count") > 1)
        .sum().alias("true_label_conflict_ids"),

    (
        (pl.col("valid_label_count") == 1)
        & pl.col("has_missing_label")
    ).sum().alias("valid_label_with_missing_ids"),

    (pl.col("valid_label_count") == 0)
        .sum().alias("all_labels_missing_ids")
]).collect()

display(label_conflict_summary)

true_label_conflict_ids,valid_label_with_missing_ids,all_labels_missing_ids
u32,u32,u32
43,58,322


In [33]:
true_label_conflict_samples = (
    label_conflict_audit
    .filter(pl.col("valid_label_count") > 1)
    .select(["id", "valid_labels"])
    .join(
        comments_lf,
        on="id",
        how="inner"
    )
    .select([
        "id",
        "body",
        "rate",
        "recommendation_status",
        "product_id"
    ])
    .sort("id")
    .collect()
)

display(true_label_conflict_samples.head(20))

id,body,rate,recommendation_status,product_id
i64,str,f64,str,i64
24565793,"""قدرت چسبندگی خوبی داره، بد نیس…",3.0,"""no_idea""",2233238
24565793,"""قدرت چسبندگی خوبی داره، عملکرد…",5.0,"""recommended""",2233238
33691371,"""کرم خوبیه اما تاریخ مصرفش گذشت…",4.0,"""not_recommended""",3548526
33691371,"""خیلی توصیه نمیکنمش، فوق العاده…",3.0,"""no_idea""",3548526
37394585,"""خوب نیس""",5.0,"""not_recommended""",2233954
…,…,…,…,…
46322276,"""داخل حمام به ساقه ی موهای رنگ …",3.0,"""no_idea""",5177204
46345765,"""بسته بندی شامپو نسبت به خریدای…",4.0,"""recommended""",7933056
46345765,"""بسته بندی شامپو نسبت به خریدای…",1.0,"""not_recommended""",7933056


In [34]:
complete_label_audit = label_conflict_audit.select([
    pl.len().alias("total_duplicate_ids"),

    (pl.col("valid_label_count") > 1)
        .sum().alias("true_conflicts"),

    (
        (pl.col("valid_label_count") == 1)
        & pl.col("has_missing_label")
    ).sum().alias("valid_with_missing"),

    (pl.col("valid_label_count") == 0)
        .sum().alias("all_missing"),

    (
        (pl.col("valid_label_count") == 1)
        & ~pl.col("has_missing_label")
    ).sum().alias("consistent_valid_label")
]).collect()

display(complete_label_audit)

total_duplicate_ids,true_conflicts,valid_with_missing,all_missing,consistent_valid_label
u32,u32,u32,u32,u32
3226,43,58,322,2803


In [35]:
duplicate_bodies = (
    comments_lf
    .filter(
        pl.col("body").is_not_null()
        & (pl.col("body").str.strip_chars() != "")
        & (
            pl.col("body").str.to_lowercase()
            .str.strip_chars()
            .is_in(["nan", "null", "none", "n/a"])
            .not_()
        )
    )
    .group_by("body")
    .agg([
        pl.len().alias("occurrences"),
        pl.col("id").n_unique().alias("unique_ids"),
        pl.col("recommendation_status")
            .n_unique().alias("label_variants")
    ])
    .filter(pl.col("occurrences") > 1)
)

duplicate_body_summary = duplicate_bodies.select([
    pl.len().alias("duplicated_body_groups"),
    pl.col("occurrences").sum().alias("rows_in_duplicate_groups"),
    (pl.col("occurrences") - 1)
        .sum().alias("extra_duplicate_body_rows"),
    (pl.col("unique_ids") > 1)
        .sum().alias("bodies_with_multiple_ids"),
    (pl.col("label_variants") > 1)
        .sum().alias("bodies_with_multiple_raw_labels")
]).collect()

display(duplicate_body_summary)

duplicated_body_groups,rows_in_duplicate_groups,extra_duplicate_body_rows,bodies_with_multiple_ids,bodies_with_multiple_raw_labels
u32,u32,u32,u32,u32
197744,2217139,2019395,195613,75364


In [36]:
duplicate_body_label_audit = (
    comments_lf
    .with_columns(
        pl.when(
            pl.col("recommendation_status")
            .str.strip_chars()
            .str.to_lowercase()
            .is_in(["recommended", "not_recommended", "no_idea"])
        )
        .then(
            pl.col("recommendation_status")
            .str.strip_chars()
            .str.to_lowercase()
        )
        .otherwise(None)
        .alias("clean_label")
    )
    .group_by("body")
    .agg([
        pl.len().alias("occurrences"),
        pl.col("id").n_unique().alias("unique_ids"),
        pl.col("clean_label").drop_nulls()
            .n_unique().alias("valid_label_count"),
        pl.col("clean_label").is_null()
            .any().alias("has_missing_label")
    ])
    .filter(pl.col("occurrences") > 1)
)

duplicate_body_label_summary = (
    duplicate_body_label_audit
    .select([
        pl.len().alias("duplicate_body_groups"),

        (pl.col("valid_label_count") > 1)
            .sum().alias("true_label_conflicts"),

        (
            (pl.col("valid_label_count") == 1)
            & pl.col("has_missing_label")
        ).sum().alias("one_valid_label_with_missing"),

        (pl.col("valid_label_count") == 0)
            .sum().alias("all_labels_missing"),

        (
            (pl.col("valid_label_count") == 1)
            & ~pl.col("has_missing_label")
        ).sum().alias("consistent_label_groups")
    ])
    .collect()
)

display(duplicate_body_label_summary)

duplicate_body_groups,true_label_conflicts,one_valid_label_with_missing,all_labels_missing,consistent_label_groups
u32,u32,u32,u32,u32
197745,31942,43423,7061,115319


In [37]:
conflict_body_combinations = (
    comments_lf
    .with_columns(
        pl.when(
            pl.col("recommendation_status")
            .str.strip_chars()
            .str.to_lowercase()
            .is_in(["recommended", "not_recommended", "no_idea"])
        )
        .then(
            pl.col("recommendation_status")
            .str.strip_chars()
            .str.to_lowercase()
        )
        .otherwise(None)
        .alias("clean_label")
    )
    .group_by("body")
    .agg(
        pl.col("clean_label")
        .drop_nulls()
        .unique()
        .sort()
        .alias("labels")
    )
    .filter(pl.col("labels").list.len() > 1)
    .group_by("labels")
    .agg(
        pl.len().alias("body_groups")
    )
    .sort("body_groups", descending=True)
    .collect()
)

display(conflict_body_combinations)

labels,body_groups
list[str],u32
"[""no_idea"", ""recommended""]",20679
"[""no_idea"", ""not_recommended""]",4806
"[""no_idea"", ""not_recommended"", ""recommended""]",3928
"[""not_recommended"", ""recommended""]",2529


In [38]:
date_audit = (
    comments_lf
    .with_columns(
        pl.col("created_at")
        .str.to_datetime(strict=False)
        .alias("parsed_date")
    )
    .select([
        pl.len().alias("total_rows"),
        pl.col("parsed_date").is_null()
            .sum().alias("invalid_dates"),
        pl.col("parsed_date").min()
            .alias("earliest_date"),
        pl.col("parsed_date").max()
            .alias("latest_date"),
        pl.col("parsed_date").n_unique()
            .alias("unique_dates")
    ])
    .collect()
)

display(date_audit)

total_rows,invalid_dates,earliest_date,latest_date,unique_dates
u32,u32,datetime[μs],datetime[μs],u32
6156289,6156289,null,null,1


In [39]:
date_samples = (
    comments_lf
    .select("created_at")
    .head(30)
    .collect()
)

display(date_samples)

created_at
str
"""23 شهریور 1402"""
"""16 تیر 1399"""
"""26 مرداد 1401"""
"""28 اسفند 1399"""
"""12 شهریور 1402"""
…
"""17 مرداد 1402"""
"""13 آبان 1400"""
"""3 بهمن 1400"""


In [40]:
date_format_profile = (
    comments_lf
    .select([
        pl.col("created_at").str.len_chars()
        .alias("string_length")
    ])
    .group_by("string_length")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

display(date_format_profile)

string_length,count
u32,u32
13,1389248
11,1356763
12,1043950
10,778948
14,614070
15,495043
16,356426
9,121841


In [41]:
date_parts = (
    comments_lf
    .with_columns([
        pl.col("created_at")
        .str.split(" ")
        .alias("date_parts")
    ])
    .with_columns([
        pl.col("date_parts").list.get(0).alias("day"),
        pl.col("date_parts").list.get(1).alias("month"),
        pl.col("date_parts").list.get(2).alias("year")
    ])
)

# توزیع سال‌ها
year_distribution = (
    date_parts
    .group_by("year")
    .agg(pl.len().alias("count"))
    .sort("year")
    .collect()
)

display(year_distribution)

year,count
str,u32
"""1395""",3575
"""1396""",6357
"""1397""",16693
"""1398""",138645
"""1399""",684321
"""1400""",1253484
"""1401""",2165894
"""1402""",1887320


In [42]:
month_distribution = (
    date_parts
    .group_by("month")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

display(month_distribution)

month,count
str,u32
"""شهریور""",683501
"""تیر""",581390
"""مرداد""",580158
"""خرداد""",569773
"""مهر""",569101
…,…
"""بهمن""",474141
"""فروردین""",468339
"""دی""",434902


In [43]:
valid_months = [
    "فروردین", "اردیبهشت", "خرداد",
    "تیر", "مرداد", "شهریور",
    "مهر", "آبان", "آذر",
    "دی", "بهمن", "اسفند"
]

date_validation = (
    date_parts
    .with_columns([
        pl.col("day").cast(pl.Int32, strict=False).alias("day_num"),
        pl.col("year").cast(pl.Int32, strict=False).alias("year_num")
    ])
    .with_columns([
        (
            pl.col("day_num").is_not_null()
            & pl.col("year_num").is_not_null()
            & pl.col("month").is_in(valid_months)
            & (pl.col("day_num") >= 1)
            & (pl.col("year_num") >= 1300)
            & (pl.col("year_num") <= 1500)
            & (
                (
                    pl.col("month").is_in([
                        "فروردین", "اردیبهشت", "خرداد",
                        "تیر", "مرداد", "شهریور"
                    ])
                    & (pl.col("day_num") <= 31)
                )
                | (
                    pl.col("month").is_in([
                        "مهر", "آبان", "آذر",
                        "دی", "بهمن"
                    ])
                    & (pl.col("day_num") <= 30)
                )
                | (
                    (pl.col("month") == "اسفند")
                    & (pl.col("day_num") <= 30)
                )
            )
        ).alias("structurally_valid")
    ])
)

date_validation_summary = (
    date_validation
    .select([
        pl.len().alias("total_rows"),
        pl.col("structurally_valid")
            .sum().alias("structurally_valid_rows"),
        (~pl.col("structurally_valid"))
            .sum().alias("structurally_invalid_rows")
    ])
    .collect()
)

display(date_validation_summary)

total_rows,structurally_valid_rows,structurally_invalid_rows
u32,u32,u32
6156289,6156289,0


In [44]:
persian_text_audit = (
    text_lf
    .with_columns([
        pl.col("body")
        .str.contains(r"[\u0600-\u06FF]")
        .alias("has_persian"),

        pl.col("body")
        .str.contains(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
        .alias("has_control_chars"),

        pl.col("body")
        .str.contains(r"[\u200B\u200C\u200D\uFEFF]")
        .alias("has_invisible_chars")
    ])
    .select([
        pl.len().alias("total_valid_texts"),

        pl.col("has_persian")
        .sum().alias("texts_with_persian"),

        (~pl.col("has_persian"))
        .sum().alias("texts_without_persian"),

        pl.col("has_control_chars")
        .sum().alias("texts_with_control_chars"),

        pl.col("has_invisible_chars")
        .sum().alias("texts_with_invisible_chars")
    ])
    .collect()
)

display(persian_text_audit)

total_valid_texts,texts_with_persian,texts_without_persian,texts_with_control_chars,texts_with_invisible_chars
u32,u32,u32,u32,u32
6155652,6151410,4242,1,194509


In [45]:
invisible_char_profile = (
    text_lf
    .select([
        pl.col("body").str.contains("\u200c")
            .sum().alias("zero_width_non_joiner"),

        pl.col("body").str.contains("\u200d")
            .sum().alias("zero_width_joiner"),

        pl.col("body").str.contains("\u200b")
            .sum().alias("zero_width_space"),

        pl.col("body").str.contains("\ufeff")
            .sum().alias("byte_order_mark")
    ])
    .collect()
)

display(invisible_char_profile)

zero_width_non_joiner,zero_width_joiner,zero_width_space,byte_order_mark
u32,u32,u32,u32
193672,911,7,2


In [46]:
non_persian_samples = (
    text_lf
    .filter(
        ~pl.col("body").str.contains(r"[\u0600-\u06FF]")
    )
    .select("body")
    .head(30)
    .collect()
)

display(non_persian_samples)

body
str
"""Good book"""
"""Khooob boood"""
"""mahsouli khub ba gheimate mona…"
"""raziam"""
"""ok bud"""
…
"""Really good"""
"""Awli bood"""
"""Good book"""


In [47]:
control_char_samples = (
    text_lf
    .filter(
        pl.col("body").str.contains(
            r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]"
        )
    )
    .select("body")
    .collect()
)

display(control_char_samples)

body
str
"""-از لحاظ جنسش به نظرم خیلی خو…"


In [48]:
control_texts = control_char_samples["body"].to_list()

for text in control_texts:
    print("Raw representation:", repr(text))

    suspicious_chars = [
        (char, f"U+{ord(char):04X}")
        for char in text
        if ord(char) < 32 or ord(char) == 127
    ]

    print("Control characters:", suspicious_chars)

Raw representation: '-\x16از لحاظ جنسش به نظرم خیلی خوب بود، رنگ و شکلش دقیقا مثل عکسش بود.\r\n\r\n-فقط به نظر من تن خورش جالب نبود که اونم شاید به سلیقه ربط داشته باشه به هر حال من مرجوع کردم.'
Control characters: [('\x16', 'U+0016'), ('\r', 'U+000D'), ('\n', 'U+000A'), ('\r', 'U+000D'), ('\n', 'U+000A')]


In [49]:
numeric_comments_audit = comments_lf.select([
    pl.len().alias("total_rows"),

    pl.col("id").min().alias("id_min"),
    pl.col("id").max().alias("id_max"),
    pl.col("id").null_count().alias("id_nulls"),
    (pl.col("id") < 0).sum().alias("negative_ids"),

    pl.col("rate").min().alias("rate_min"),
    pl.col("rate").max().alias("rate_max"),
    pl.col("rate").null_count().alias("rate_nulls"),
    (pl.col("rate") < 0).sum().alias("negative_rates"),
    (pl.col("rate") > 5).sum().alias("rates_above_5"),

    pl.col("product_id").min().alias("product_id_min"),
    pl.col("product_id").max().alias("product_id_max"),
    pl.col("product_id").null_count().alias("product_id_nulls"),
    (pl.col("product_id") <= 0).sum().alias("nonpositive_product_ids"),

    pl.col("likes").min().alias("likes_min"),
    pl.col("likes").max().alias("likes_max"),
    pl.col("likes").null_count().alias("likes_nulls"),
    (pl.col("likes") < 0).sum().alias("negative_likes"),

    pl.col("dislikes").min().alias("dislikes_min"),
    pl.col("dislikes").max().alias("dislikes_max"),
    pl.col("dislikes").null_count().alias("dislikes_nulls"),
    (pl.col("dislikes") < 0).sum().alias("negative_dislikes")
]).collect()

display(numeric_comments_audit)

total_rows,id_min,id_max,id_nulls,negative_ids,rate_min,rate_max,rate_nulls,negative_rates,rates_above_5,product_id_min,product_id_max,product_id_nulls,nonpositive_product_ids,likes_min,likes_max,likes_nulls,negative_likes,dislikes_min,dislikes_max,dislikes_nulls,negative_dislikes
u32,i64,i64,u32,u32,f64,f64,u32,u32,u32,i64,i64,u32,u32,i64,i64,u32,u32,i64,i64,u32,u32
6156289,981586,54745160,0,0,0.0,2500.0,0,0,1,7602,12702212,0,0,0,1136,0,0,0,336,0,0


In [50]:
numeric_products_audit = products_lf.select([
    pl.len().alias("total_rows"),

    pl.col("id").min().alias("id_min"),
    pl.col("id").max().alias("id_max"),
    pl.col("id").null_count().alias("id_nulls"),
    (pl.col("id") <= 0).sum().alias("nonpositive_ids"),

    pl.col("Rate").min().alias("rate_min"),
    pl.col("Rate").max().alias("rate_max"),
    pl.col("Rate").null_count().alias("rate_nulls"),

    pl.col("Rate_cnt").min().alias("rate_cnt_min"),
    pl.col("Rate_cnt").max().alias("rate_cnt_max"),
    pl.col("Rate_cnt").null_count().alias("rate_cnt_nulls"),
    (pl.col("Rate_cnt") < 0).sum().alias("negative_rate_counts"),

    pl.col("Price").min().alias("price_min"),
    pl.col("Price").max().alias("price_max"),
    pl.col("Price").null_count().alias("price_nulls"),
    (pl.col("Price") < 0).sum().alias("negative_prices"),

    pl.col("min_price_last_month").min()
        .alias("min_price_last_month_min"),
    pl.col("min_price_last_month").max()
        .alias("min_price_last_month_max"),
    pl.col("min_price_last_month").null_count()
        .alias("min_price_last_month_nulls"),
    (pl.col("min_price_last_month") < 0)
        .sum().alias("negative_min_prices")
]).collect()

display(numeric_products_audit)

total_rows,id_min,id_max,id_nulls,nonpositive_ids,rate_min,rate_max,rate_nulls,rate_cnt_min,rate_cnt_max,rate_cnt_nulls,negative_rate_counts,price_min,price_max,price_nulls,negative_prices,min_price_last_month_min,min_price_last_month_max,min_price_last_month_nulls,negative_min_prices
u32,i64,i64,u32,u32,i64,i64,u32,i64,i64,u32,u32,i64,i64,u32,u32,i64,i64,u32,u32
1283496,7602,12773763,0,0,0,100,0,0,30438,0,0,0,8499990000,0,0,0,7645500000,0,0


In [51]:
zero_value_audit = products_lf.select([
    pl.len().alias("total_products"),

    (pl.col("Rate") == 0)
        .sum().alias("zero_rate_count"),

    (pl.col("Rate_cnt") == 0)
        .sum().alias("zero_rate_cnt_count"),

    (pl.col("Price") == 0)
        .sum().alias("zero_price_count"),

    (pl.col("min_price_last_month") == 0)
        .sum().alias("zero_min_price_count"),

    (pl.col("Price") > 1_000_000_000)
        .sum().alias("price_above_1b_count"),

    (pl.col("min_price_last_month") > 1_000_000_000)
        .sum().alias("min_price_above_1b_count")
]).collect()

display(zero_value_audit)

total_products,zero_rate_count,zero_rate_cnt_count,zero_price_count,zero_min_price_count,price_above_1b_count,min_price_above_1b_count
u32,u32,u32,u32,u32,u32,u32
1283496,799172,799151,223,1208195,303,8


In [52]:
price_distribution = products_lf.select([
    pl.col("Price").mean().alias("price_mean"),
    pl.col("Price").median().alias("price_median"),
    pl.col("Price").quantile(0.90).alias("price_p90"),
    pl.col("Price").quantile(0.95).alias("price_p95"),
    pl.col("Price").quantile(0.99).alias("price_p99"),

    pl.col("min_price_last_month").mean()
        .alias("min_price_mean"),

    pl.col("min_price_last_month").median()
        .alias("min_price_median"),

    pl.col("min_price_last_month").quantile(0.99)
        .alias("min_price_p99")
]).collect()

display(price_distribution)

price_mean,price_median,price_p90,price_p95,price_p99,min_price_mean,min_price_median,min_price_p99
f64,f64,f64,f64,f64,f64,f64,f64
8.4913e6,1.56e6,1.5e7,3.30662e7,1.1e8,421214.718359,0.0,6.39e6


In [53]:
product_consistency_audit = products_lf.select([
    # رابطه امتیاز و تعداد رأی
    (
        (pl.col("Rate") == 0)
        & (pl.col("Rate_cnt") == 0)
    ).sum().alias("both_rate_and_count_zero"),

    (
        (pl.col("Rate") == 0)
        & (pl.col("Rate_cnt") > 0)
    ).sum().alias("zero_rate_with_positive_count"),

    (
        (pl.col("Rate") > 0)
        & (pl.col("Rate_cnt") == 0)
    ).sum().alias("positive_rate_with_zero_count"),

    (
        (pl.col("Rate") > 0)
        & (pl.col("Rate_cnt") > 0)
    ).sum().alias("both_positive"),

    # رابطه قیمت فعلی و قیمت ماه گذشته
    (
        (pl.col("Price") == 0)
        & (pl.col("min_price_last_month") > 0)
    ).sum().alias("zero_current_price_positive_history"),

    (
        (pl.col("Price") > 0)
        & (pl.col("min_price_last_month") == 0)
    ).sum().alias("positive_current_price_zero_history"),

    (
        (pl.col("min_price_last_month") > pl.col("Price"))
        & (pl.col("Price") > 0)
    ).sum().alias("historical_min_above_current_price"),

    (
        (pl.col("min_price_last_month") > 0)
        & (pl.col("Price") > 0)
        & (pl.col("min_price_last_month") <= pl.col("Price"))
    ).sum().alias("historical_min_below_or_equal_current")
]).collect()

display(product_consistency_audit)

both_rate_and_count_zero,zero_rate_with_positive_count,positive_rate_with_zero_count,both_positive,zero_current_price_positive_history,positive_current_price_zero_history,historical_min_above_current_price,historical_min_below_or_equal_current
u32,u32,u32,u32,u32,u32,u32,u32
799151,21,0,484324,22,1207994,7093,68186


In [54]:
label_quality_audit = (
    comments_lf
    .with_columns(
        pl.col("recommendation_status")
        .str.strip_chars()
        .str.to_lowercase()
        .alias("clean_label")
    )
    .with_columns(
        pl.when(
            pl.col("clean_label").is_in(
                ["nan", "null", "none", "n/a", ""]
            )
        )
        .then(None)
        .otherwise(pl.col("clean_label"))
        .alias("clean_label")
    )
    .with_columns(
        pl.when(
            pl.col("clean_label").is_in([
                "recommended",
                "not_recommended",
                "no_idea"
            ])
        )
        .then(pl.lit("valid"))
        .when(pl.col("clean_label").is_null())
        .then(pl.lit("missing"))
        .otherwise(pl.lit("unexpected"))
        .alias("label_status")
    )
    .group_by(["clean_label", "label_status"])
    .agg(pl.len().alias("count"))
    .sort("label_status", "count", descending=[False, True])
    .collect()
)

display(label_quality_audit)

clean_label,label_status,count
str,str,u32
null,"""missing""",894426
"""recommended""","""valid""",4162839
"""no_idea""","""valid""",594121
"""not_recommended""","""valid""",504903


In [55]:
rating_band_audit = (
    comments_lf
    .with_columns(
        pl.when(
            (pl.col("rate") >= 0)
            & (pl.col("rate") < 2.5)
        )
        .then(pl.lit("low"))

        .when(
            (pl.col("rate") >= 2.5)
            & (pl.col("rate") < 4)
        )
        .then(pl.lit("medium"))

        .when(
            (pl.col("rate") >= 4)
            & (pl.col("rate") <= 5)
        )
        .then(pl.lit("high"))

        .otherwise(pl.lit("invalid"))
        .alias("rating_band")
    )
    .group_by("rating_band")
    .agg(pl.len().alias("count"))
    .sort("rating_band")
    .collect()
)

display(rating_band_audit)

rating_band,count
str,u32
"""high""",4004905
"""invalid""",1
"""low""",1092329
"""medium""",1059054


In [56]:
boundary_audit = comments_lf.select([
    (pl.col("rate") == 2.5)
        .sum().alias("rate_equal_2_5"),

    (pl.col("rate") == 4.0)
        .sum().alias("rate_equal_4"),

    (pl.col("rate") == 0)
        .sum().alias("rate_equal_0"),

    (pl.col("rate") == 5)
        .sum().alias("rate_equal_5"),

    (pl.col("rate") > 5)
        .sum().alias("rate_above_5"),

    (pl.col("rate") < 0)
        .sum().alias("rate_below_0")
]).collect()

display(boundary_audit)

rate_equal_2_5,rate_equal_4,rate_equal_0,rate_equal_5,rate_above_5,rate_below_0
u32,u32,u32,u32,u32,u32
5438,1515723,529098,2398801,1,0


In [57]:
label_clean_expr = (
    pl.when(
        pl.col("recommendation_status")
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(["nan", "null", "none", "n/a", ""])
    )
    .then(None)
    .otherwise(
        pl.col("recommendation_status")
        .str.strip_chars()
        .str.to_lowercase()
    )
)

missing_by_buyer = (
    comments_lf
    .with_columns(label_clean_expr.alias("clean_label"))
    .group_by("is_buyer")
    .agg([
        pl.len().alias("total"),
        pl.col("clean_label").is_null().sum().alias("missing_labels")
    ])
    .with_columns(
        (pl.col("missing_labels") / pl.col("total") * 100)
        .round(3)
        .alias("missing_percentage")
    )
    .collect()
)

display(missing_by_buyer)

is_buyer,total,missing_labels,missing_percentage
bool,u32,u32,f64
true,5854108,592245,10.117
false,302181,302181,100.0


In [58]:
(
    comments_lf
    .group_by("true_to_size_rate")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(30)
    .collect()
)

true_to_size_rate,count
str,u32
"""nan""",6065516
"""fit""",70622
"""small""",9096
"""big""",5626
"""tooSmall""",3438
"""tooBig""",1991
